In [0]:
# Databricks notebook source

import json
from pyspark.sql.functions import current_timestamp, lit

# ============================================================
# 1) READ CONFIG
# ============================================================

dbutils.widgets.text("config_path", "")
config_path = dbutils.widgets.get("config_path").strip()

if not config_path:
    raise ValueError("config_path is required")

print("CONFIG PATH:", config_path)

config_text = "\n".join(
    row.value for row in spark.read.text(config_path).collect()
)
config = json.loads(config_text)

print("CONFIG KEYS:", list(config.keys()))
print("CONFIG LOADED:", config["entity_name"])

# ============================================================
# 2) KAFKA OPTIONS
# ============================================================

if "secret_scope" in config and "secret_key" in config:
    connection_string = dbutils.secrets.get(
        scope=config["secret_scope"],
        key=config["secret_key"]
    )
else:
    connection_string = config["connection_string"]

kafka_options = {
    "kafka.bootstrap.servers": config["bootstrap_servers"],
    "subscribe": config["eventhub_name"],
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.sasl.jaas.config": (
        f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
        f'username="$ConnectionString" password="{connection_string}";'
    ),
    "startingOffsets": "earliest",
    "failOnDataLoss": "false"
}

# ============================================================
# 3) READ STREAM
# ============================================================

raw_df = (
    spark.readStream
        .format("kafka")
        .options(**kafka_options)
        .load()
)

print("Kafka stream connected")

# ============================================================
# 4) BRONZE TRANSFORMATION
# ============================================================

bronze_df = (
    raw_df
        .withColumn("ingest_ts", current_timestamp())
        .withColumn("entity_name", lit(config["entity_name"]))
        .withColumn("source_name", lit(config["source_name"]))
)

print("Bronze DF ready")

# ============================================================
# 5) WRITE TO DELTA
# ============================================================
import traceback

try:
    query = (
        bronze_df.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", config["checkpoint_path"])
        .trigger(once=True)
        .start(config["bronze_path"])
    )

    print(f"Stream started -> dbx_dev.stratus.{config['entity_name']}_bronze")
    print(f"Checkpoint -> {config['checkpoint_path']}")

    query.awaitTermination()

    dbutils.notebook.exit("BRONZE SUCCESS")

except Exception as e:
    err = traceback.format_exc()
    print("BRONZE FAILED TRACEBACK:")
    print(err)
    raise

In [0]:
dbutils.fs.rm("/Volumes/dbx_dev/stratus/eventhub_volume/bronze/customer/", True)

True

In [0]:
bronze_path = "/Volumes/dbx_dev/stratus/eventhub_volume/bronze/customer/"

In [0]:
display(dbutils.fs.ls(bronze_path))

path,name,size,modificationTime
dbfs:/Volumes/dbx_dev/stratus/eventhub_volume/bronze/customer/_delta_log/,_delta_log/,0,1776152464000
dbfs:/Volumes/dbx_dev/stratus/eventhub_volume/bronze/customer/part-00000-db3d80db-5952-48ed-ad9f-4ec3a1f1816d.c000.snappy.parquet,part-00000-db3d80db-5952-48ed-ad9f-4ec3a1f1816d.c000.snappy.parquet,2889,1776152465000
